# Data Cleaning Demo — `github_repos_raw.txt`

**MGS 3001 WHS01 · Week 11-2 · 7 May 2026**

This notebook walks through the 6-step data cleaning pipeline using the hypothetical GitHub repos dataset.

**Pipeline:**  
1. Inspect → 2. Diagnose → 3. Handle Missing Values → 4. Remove Duplicates → 5. Fix Data Types → 6. Validate & Save

---

In [ ]:
import pandas as pd
import numpy as np

print(f"pandas version: {pd.__version__}")

---
## Step 1: Inspect — Load & First Look

Load the raw data and get a feel for its shape, column types, and first few rows.

In [ ]:
# Load the raw dataset
# Note: the file is .txt but is actually comma-separated (CSV format)
df = pd.read_csv("github_repos_raw.txt")

print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
df.head()

In [ ]:
# The diagnostic report — column names, types, non-null counts
df.info()

In [ ]:
# Summary statistics for numeric columns
df.describe()

**Observations from Step 1:**
- 40+ rows, 14 columns
- `stargazers_count`, `forks_count`, `open_issues_count` may be loaded as **object** (string) — something is wrong
- Several columns have fewer non-null counts than expected, indicating missing values
- `describe()` only shows columns Pandas recognized as numeric — the rest need type conversion

---

## Step 2: Diagnose — How Bad Is It?

Count missing values, duplicates, and inspect suspicious columns.

In [ ]:
# Missing values per column — count and percentage
missing = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_pct": (df.isnull().mean() * 100).round(1)
})
missing[missing["missing_count"] > 0]

In [ ]:
# Are there hidden missing values disguised as strings?
print("Unique values in open_issues_count:")
print(df["open_issues_count"].unique())
print()
print("Rows where open_issues_count is 'N/A':")
print(df[df["open_issues_count"] == "N/A"][["full_name", "open_issues_count"]])

In [ ]:
# Count duplicates
print(f"Exact duplicates: {df.duplicated().sum()}")
print(f"Duplicates by full_name: {df.duplicated(subset=['full_name']).sum()}")
print()
# Show the duplicated full_names
dup_names = df[df.duplicated(subset=["full_name"], keep=False)].sort_values("full_name")
print(f"Rows involved in full_name duplicates: {len(dup_names)}")
dup_names[["full_name", "stargazers_count", "search_keyword"]]

In [ ]:
# Check for inconsistent capitalization
print("Unique owner_type values:")
print(df["owner_type"].unique())
print()
print("Unique language values (sample):")
print(sorted(df["language"].dropna().unique()))

In [ ]:
# Check for inconsistent date formats
non_iso_updated = df[~df["updated_at"].str.contains("T", na=False)][["full_name", "updated_at"]]
non_iso_created = df[~df["created_at"].str.contains("T", na=False)][["full_name", "created_at"]]

print("Rows with non-ISO updated_at:")
print(non_iso_updated)
print()
print("Rows with non-ISO created_at:")
print(non_iso_created)

In [ ]:
# Check for unrealistic values
issues_numeric = pd.to_numeric(df["open_issues_count"], errors="coerce")
print("Negative open_issues_count:")
print(df[issues_numeric < 0][["full_name", "open_issues_count"]])

**Diagnosis Summary:**

| Issue | Count | Details |
|---|---|---|
| True missing (NaN) | ~6 cells | language, description, license_name, topics, stargazers_count, forks_count |
| Hidden missing ("N/A") | 2 rows | open_issues_count |
| Key-based duplicates | 4 extra rows | microsoft/autogen, langchain-ai/langchain, openai/openai-python, meta-llama/llama |
| Inconsistent caps | 4 rows | owner_type: "user"/"organization"; language: "python" |
| Bad date format | 3 rows | MM/DD/YYYY, date-only strings |
| Unrealistic value | 1 row | open_issues_count = -15 |

---

## Step 3: Handle Missing Values

Strategy:
- Replace hidden missing values ("N/A", empty strings) with `NaN`
- **Numeric columns**: convert to numeric, fix unrealistic values, then fill or flag
- **Categorical columns** (`language`, `license_name`): fill with a descriptive label
- **Text columns** (`description`, `topics`): fill with empty string

In [ ]:
# Step 3a: Replace hidden missing values with NaN
df = df.replace(["N/A", "n/a", ""], pd.NA)

print("After replacing hidden missing values:")
print(df.isnull().sum())
print(f"\nTotal NaN cells: {df.isnull().sum().sum()}")

In [ ]:
# Step 3b: Convert numeric columns — coerce errors to NaN
numeric_cols = ["stargazers_count", "forks_count", "open_issues_count"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("After numeric conversion:")
print(df[numeric_cols].describe())

In [ ]:
# Step 3c: Fix unrealistic values — negative open_issues_count
# Decision: replace with NaN (we don't know the true value)
mask_negative = df["open_issues_count"] < 0
print(f"Rows with negative open_issues_count: {mask_negative.sum()}")
df.loc[mask_negative, "open_issues_count"] = np.nan

In [ ]:
# Step 3d: Fill missing values with appropriate defaults

# MNAR flag: create binary indicator BEFORE filling (for regression later)
df["stars_missing"] = df["stargazers_count"].isnull().astype(int)

# Numeric: fill with median (robust to outliers) or 0 where appropriate
df["open_issues_count"] = df["open_issues_count"].fillna(df["open_issues_count"].median())
df["stargazers_count"] = df["stargazers_count"].fillna(0)
df["forks_count"] = df["forks_count"].fillna(0)

# Categorical: fill with descriptive label
df["language"] = df["language"].fillna("Not specified")
df["license_name"] = df["license_name"].fillna("No license")

# Text: fill with empty string
df["description"] = df["description"].fillna("")
df["topics"] = df["topics"].fillna("")

print("Remaining missing values:")
remaining = df.isnull().sum()
if remaining.sum() == 0:
    print("✅ No missing values remain.")
else:
    print(remaining[remaining > 0])

**Decisions documented (for the Methods section):**
- Replaced "N/A" strings with NaN before processing
- Converted `open_issues_count` = -15 to NaN (unrealistic value)
- Created `stars_missing` binary flag before filling `stargazers_count` (MNAR indicator)
- Filled `open_issues_count` with column median; `stargazers_count` and `forks_count` with 0
- Filled `language` and `license_name` with descriptive labels
- Filled `description` and `topics` with empty strings

---

## Step 4: Remove Duplicates

We have repos that appear multiple times — either from overlapping search keywords or from being collected at slightly different times with different star/fork counts.

**Decision:** deduplicate by `full_name`, keeping the first occurrence.

In [ ]:
rows_before = len(df)
print(f"Rows before dedup: {rows_before}")
print(f"Unique full_name values: {df['full_name'].nunique()}")
print()

# Remove duplicates by full_name, keeping the first occurrence
df = df.drop_duplicates(subset=["full_name"], keep="first")

print(f"Rows after dedup: {len(df)}")
print(f"Rows removed: {rows_before - len(df)}")

---
## Step 5: Fix Data Types

Three issues to fix:
1. **Capitalization** — standardize before converting to categories
2. **Date columns** — convert strings to datetime
3. **Numeric columns** — ensure proper integer type

In [ ]:
# Step 5a: Fix inconsistent capitalization
# str.title() capitalizes first letter of each word
# "python" → "Python", "organization" → "Organization"

df["language"] = df["language"].str.strip().str.title()
df["owner_type"] = df["owner_type"].str.strip().str.title()

print("Unique language values after standardization:")
print(sorted(df["language"].unique()))
print()
print("Unique owner_type values after standardization:")
print(sorted(df["owner_type"].unique()))

In [ ]:
# Step 5b: Convert date columns to datetime
# Our data has mixed formats:
#   - ISO 8601:   2023-09-19T17:06:55Z
#   - MM/DD/YYYY: 02/14/2025
#   - Date only:  2024-03-15
# pd.to_datetime with format="mixed" handles all of these

for col in ["created_at", "updated_at"]:
    df[col] = pd.to_datetime(df[col], format="mixed", utc=True)

print("Sample dates after conversion:")
print(df[["full_name", "created_at", "updated_at"]].head())
print(f"\ncreated_at dtype: {df['created_at'].dtype}")
print(f"updated_at dtype: {df['updated_at'].dtype}")

In [ ]:
# Step 5c: Derive new variables from cleaned dates
now = pd.Timestamp.now(tz="UTC")

df["repo_age_days"] = (now - df["created_at"]).dt.days
df["days_since_update"] = (now - df["updated_at"]).dt.days

print("New derived columns:")
print(df[["full_name", "repo_age_days", "days_since_update"]].head())

In [ ]:
# Step 5d: Final type conversions
# Convert owner_type to categorical
df["owner_type"] = df["owner_type"].astype("category")

# Convert numeric columns to Int64 (nullable integer — handles any stray NaN gracefully)
for col in ["stargazers_count", "forks_count", "open_issues_count"]:
    df[col] = df[col].astype("Int64")

print("Final dtypes:")
print(df.dtypes)

---
## Step 6: Validate & Save

Re-run diagnostics to confirm all issues are resolved, then save the cleaned dataset.

In [ ]:
# Final validation
print("=" * 55)
print("  FINAL VALIDATION REPORT")
print("=" * 55)
print(f"  Rows:              {len(df)}  (started with {rows_before})")
print(f"  Columns:           {len(df.columns)}  (started with 14, added 3 derived)")
print(f"  Missing (NaN):     {df.isnull().sum().sum()}")
print(f"  Dup (full_name):   {df.duplicated(subset=['full_name']).sum()}")
print(f"  Negative issues:   {(df['open_issues_count'] < 0).sum()}")
print(f"  owner_type values: {sorted(df['owner_type'].unique())}")
print(f"  Date dtype:        {df['created_at'].dtype}")
print("=" * 55)

In [ ]:
# Quick sanity check on distributions
print("owner_type distribution:")
print(df["owner_type"].value_counts())
print()
print("Top 5 languages:")
print(df["language"].value_counts().head())
print()
print("Stars summary:")
print(df["stargazers_count"].describe())

In [ ]:
# Save the cleaned dataset
output_file = "github_repos_cleaned.csv"
df.to_csv(output_file, index=False)

print(f"✅ Saved: {output_file}")
print(f"   {len(df)} rows × {len(df.columns)} columns")

In [ ]:
# Final look at the cleaned data
df.head(10)

---
## Cleaning Summary (for your Methods section)

| Step | Action | Rows/Cells Affected |
|---|---|---|
| Replace hidden missing | Converted "N/A" and empty strings to NaN | ~4 cells |
| Fix unrealistic values | Set negative `open_issues_count` to NaN | 1 row |
| Fill numeric missing | `open_issues_count` → median; `stargazers`/`forks` → 0 + flag | ~4 cells |
| Fill categorical missing | `language` → "Not specified"; `license` → "No license" | ~5 cells |
| Fill text missing | `description`, `topics` → empty string | ~4 cells |
| Remove duplicates | Deduplicated by `full_name` (keep first) | 4 rows removed |
| Standardize capitalization | `.str.title()` on `language`, `owner_type` | 4 cells |
| Convert date types | `pd.to_datetime()` on `created_at`, `updated_at` | All rows |
| Derive variables | `repo_age_days`, `days_since_update`, `stars_missing` | 3 new columns |

**Final dataset:** ~37 rows × 17 columns (from ~41 × 14)